<a href="https://colab.research.google.com/github/farrelrassya/ML-DS-blueprints-for-finance/blob/main/Chapter01_ML_in_Finance_Landscape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 — Machine Learning in Finance: The Landscape

> *"Machine learning promises to shake up large swathes of finance."* — **The Economist** (2017)

This notebook accompanies **Chapter 1** of *Machine Learning and Data Science Blueprints for Finance*. The chapter itself is largely conceptual — a map of where machine learning is being applied across the financial industry and a taxonomy of the main families of ML algorithms. Our job here is to turn that map into something concrete.

We will:

1. Walk through the **ten application areas** where ML is reshaping finance, with the financial intuition behind each.
2. Clarify the relationships between **AI, machine learning, deep learning, and data science** — terms that are often used interchangeably but mean different things.
3. Explore the **three families of machine learning** — supervised, unsupervised, and reinforcement — by running a small finance-flavored demo for each. Every demo uses synthetic data, runs in a few seconds, and is designed to make the abstract definitions tangible.
4. Close with a brief look at **natural language processing (NLP)**, the bridge that lets ML read earnings calls, news, and analyst reports.

By the end, you should be able to answer three questions confidently: *What kind of problem am I looking at? Which ML family fits it? What would the simplest baseline look like?* These three questions will follow us through every chapter in the book.

## Setup

The cell below installs and imports everything the notebook needs. On Google Colab all of these packages are pre-installed, so `pip` finishes almost instantly. We set a global random seed so every demo is reproducible — an essential habit in quantitative work, where a result you cannot reproduce is a result you cannot trust.

In [1]:
# Install required packages (already present on Colab; this is a no-op there)
!pip install -q numpy pandas scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Global seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('Environment ready.')
print(f'NumPy:        {np.__version__}')
print(f'Pandas:       {pd.__version__}')
import sklearn
print(f'scikit-learn: {sklearn.__version__}')

Environment ready.
NumPy:        2.0.2
Pandas:       2.2.2
scikit-learn: 1.6.1


Once this cell runs we have a deterministic environment: any number we report below will be reproducible bit-for-bit by anyone running the notebook with the same library versions. This matters more in finance than in most domains — backtests that quietly vary across runs are a classic source of false alpha.

## 1. Current and Future Applications of ML in Finance

Machine learning is not a single thing being deployed in a single way. It is a toolbox that different parts of a financial institution are picking up for very different reasons. The ten areas below cover most of what you will see at a modern bank, hedge fund, or fintech firm. For each one, notice **(a) what kind of data** drives it and **(b) which ML family** is the natural fit — these two cues will help you place every new problem you encounter into the right corner of the toolbox.

### 1.1 Algorithmic Trading

**Algorithmic trading** (or *algo trading*) is the use of computer programs to execute trades autonomously, following pre-programmed instructions. Its origins go back to the 1970s, but ML has pushed it into a new regime: instead of fixed rules ("buy when the 50-day moving average crosses the 200-day"), modern systems learn rules from data and re-calibrate them in real time as the market regime shifts.

The ML edge here comes from three places. First, **speed**: an ML model can score a new market state in microseconds. Second, **breadth**: it can ingest hundreds of signals (price, volume, order-book imbalance, news sentiment, macro releases) that no human trader could weigh simultaneously. Third, **adaptivity**: when correlations break — as they did in March 2020 — a learning system can detect the regime change far faster than a rule-based one.

Most institutional players are deliberately opaque about their ML approaches, for the obvious reason that disclosed alpha is dead alpha. But the direction of travel is clear: by some industry estimates, well over half of US equity trading volume is now generated by algorithms, with ML playing a growing role inside them.

### 1.2 Portfolio Management and Robo-Advisors

**Robo-advisors** are algorithms that build and rebalance a portfolio to fit an investor's goals and risk tolerance. The user tells the system: *I am 32, I earn this much, I want to retire at 65 with \$250,000 in real terms.* The system then chooses an asset allocation, executes the trades, monitors drift, and rebalances when the portfolio strays from target weights.

Mathematically, the core task is a constrained optimization — find weights $\mathbf{w}$ that maximize expected utility subject to budget, risk, and possibly liquidity constraints:

$$\max_{\mathbf{w}} \;\; \mathbf{w}^T \boldsymbol{\mu} - \frac{\lambda}{2} \mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w} \quad \text{subject to} \quad \mathbf{1}^T \mathbf{w} = 1, \; \mathbf{w} \succeq \mathbf{0}$$

where $\boldsymbol{\mu}$ is the vector of expected returns, $\boldsymbol{\Sigma}$ is the covariance matrix, and $\lambda$ is the investor's risk aversion. Classical Markowitz theory solves this in closed form when $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ are known — but they never are. **ML enters precisely there**: estimating $\boldsymbol{\mu}$, shrinking $\boldsymbol{\Sigma}$, and personalising $\lambda$ from a user's revealed behaviour.

### 1.3 Fraud Detection

Card fraud, account-takeover, wire fraud, and synthetic identity fraud together cost the global financial system tens of billions of dollars per year. The traditional defence was a hand-crafted rule engine: *flag any transaction above \$10,000 from a new device in a foreign country.* Such rules catch the obvious cases but are brittle, easy for fraudsters to learn around, and produce a flood of false positives.

ML reframes the problem. A model is trained on hundreds of millions of labelled transactions and learns a probability:

$$P(\text{fraud} \mid \mathbf{x}) \;=\; f_\theta(\text{amount},\, \text{merchant},\, \text{device},\, \text{time},\, \text{velocity},\, \dots)$$

Crucially, the model does not need an explicit rule for every fraud pattern — it discovers them. Equally important, it can flag *anomalies* even when the pattern has never been seen before (an unsupervised use case). This is exactly the strength of ML: scanning vast datasets at speed and surfacing the small fraction that warrants a human reviewer's attention.

### 1.4 Loans, Credit Cards, and Insurance Underwriting

**Underwriting** is, in many ways, the textbook ML problem in finance. The input is a vector $\mathbf{x}$ describing the applicant — income, debt, credit history, employment, sometimes hundreds of features. The output is a binary or continuous risk estimate: *Will this person default? What probability? At what loss given default?*

The asymmetry of mistakes is what makes this domain so interesting. A false approval costs the bank the loss given default. A false rejection costs the bank a foregone customer plus, increasingly, a regulatory and reputational risk if the rejection pattern correlates with a protected attribute. The model objective is therefore never just "maximize accuracy" — it is to balance these costs while respecting fairness constraints. Logistic regression remains the regulatory default because every coefficient is auditable; gradient-boosted trees and neural networks are gaining ground where explainability tooling (SHAP, counterfactuals) is in place.

### 1.5 Automation and Chatbots

A great deal of work inside a bank is high-volume and low-judgement: reconciling two ledgers, matching a confirmation to a trade, looking up a customer's last five transactions. ML and rule-based automation together remove this work from human queues — and, because the systems learn from corrections, they get better with use.

Customer-facing chatbots are the visible tip of this iceberg. Behind the scenes, the much larger value is in **operations automation**: extracting numbers from a scanned invoice (OCR + NLP), routing a query to the right team (text classification), or proposing the journal entry for a transaction (sequence-to-sequence generation). The freed human capacity is redirected to the small fraction of cases the model is unsure about — a pattern called **human-in-the-loop**.

### 1.6 Risk Management

Risk management is the discipline of quantifying and controlling the chance of loss. Every major risk function — credit risk, market risk, liquidity risk, operational risk, model risk — now has ML somewhere in its stack.

The shift is most visible in **market risk**, where Value-at-Risk and Expected Shortfall used to be computed by historical simulation or by parametric models with strong distributional assumptions. ML brings two improvements: better tail-event estimation through quantile regression and generative models, and faster scenario generation for stress testing through neural network surrogates of slow Monte Carlo pricers.

### 1.7 Asset Price Prediction

**Asset price prediction** is the most discussed and most contested application of ML in finance. The promise is intuitive: feed the model price history, volume, fundamentals, and sentiment, and let it predict tomorrow's return. The reality is harder, for one stubborn reason — the **efficient-market hypothesis** says that any predictable pattern in prices is competed away the moment it becomes known.

That does not make the problem hopeless. It makes it a problem about *finding patterns faster than competitors* and *acting on them at a cost that leaves alpha after frictions*. Most of the book's later chapters explore exactly this — supervised regression and classification on returns, dimensionality reduction on factors, deep learning on order books, and reinforcement learning on execution.

### 1.8 Derivative Pricing

Classical derivatives pricing — the world of Black–Scholes, volatility surfaces, and Excel spreadsheet models — rests on assumptions (log-normal returns, constant volatility, frictionless markets) that everyone knows are wrong. The standard fix is to bolt patches onto the model: stochastic volatility, jumps, local-vol surfaces calibrated daily.

ML offers a different angle. Instead of starting from a stochastic process and deriving a price, train a neural network directly:

$$\text{NN}_\theta(\text{strike}, \text{maturity}, \text{spot}, \text{rate}, \text{vol surface}, \dots) \;\approx\; \text{market price}$$

The network has no idea what Itō's lemma is. It just minimizes the gap between its output and observed market prices. The trade-off is interpretability for fit — and for many exotic derivatives where no closed form exists, this is a very attractive trade. The biggest production use today is as a **fast surrogate** for slow Monte Carlo pricers in real-time risk systems.

### 1.9 Sentiment Analysis

Financial markets move on information, and most information arrives as text — news headlines, earnings call transcripts, regulatory filings, tweets, Reddit posts. **Sentiment analysis** is the family of ML techniques that turns this unstructured text into a numerical signal: how positive or negative is the market's mood toward a stock, sector, or central bank statement?

The basic pipeline is: tokenize text $\rightarrow$ embed words or sentences $\rightarrow$ classify or score. Modern systems use transformer-based language models fine-tuned on financial corpora (FinBERT is a well-known example). The output sentiment score then becomes one more feature in a downstream trading or risk model. We dedicate **Chapter 10** of this book to NLP, and several later case studies use sentiment as an input feature.

### 1.10 Trade Settlement, Money Laundering, and Beyond

Two further applications round out the picture. **Trade settlement** — the post-trade process of moving securities and cash between accounts — is mostly automated, but roughly 30% of trades still need manual intervention. ML can predict which trades are likely to fail, diagnose why, and propose a fix, compressing a 5–10 minute manual task into a fraction of a second.

**Anti-money-laundering (AML)** is a problem of scale: the UN estimates that 2–5% of global GDP is laundered each year. The needle-in-a-haystack nature of the task makes it a natural fit for unsupervised anomaly detection and graph-based ML — looking not at individual transactions but at the network of accounts they form. The financial cost of getting AML wrong is large; the regulatory cost is larger.

A consistent theme runs through all ten applications: ML is rarely replacing humans wholesale. It is **augmenting** them, taking the high-volume routine work and routing the genuinely hard cases to people. That is the realistic picture of where this technology is going for the next decade.

## 2. AI, Machine Learning, Deep Learning, and Data Science

Four terms get used almost interchangeably in everyday conversation. They are not the same thing, and a working data scientist needs to be precise about which one they are doing at any moment.

> **[Picture 1.1 — AI, machine learning, deep learning, and data science.]** The figure shows three concentric circles: **Artificial Intelligence** is the outermost (the field of making computers do things that require intelligence when done by humans). **Machine Learning** is a subset of AI (techniques that let computers identify patterns in data and improve from experience). **Deep Learning** is a subset of ML (techniques based on multi-layer neural networks). **Data Science** is a partially-overlapping discipline that *uses* ML, DL, and AI alongside statistics, big data, and cloud tools to extract insight from data.

Here is the practical distinction in one sentence each:

**Artificial Intelligence** is the *goal* — making machines perform tasks that require intelligence (vision, speech, reasoning, decision-making). It is a research field, not an algorithm.

**Machine Learning** is the *dominant approach to AI* today. Rather than hand-coding rules, you give the algorithm examples and let it learn the rules. Linear regression, logistic regression, decision trees, random forests, support vector machines, and neural networks are all ML.

**Deep Learning** is a *subset of ML* defined by the use of artificial neural networks with many stacked layers. The depth lets the network learn hierarchical representations: edges $\rightarrow$ textures $\rightarrow$ object parts $\rightarrow$ objects in vision; characters $\rightarrow$ words $\rightarrow$ phrases $\rightarrow$ meaning in language. DL has driven essentially every headline AI breakthrough of the last decade.

**Data Science** is the *practice* of extracting insight from data. A data scientist's day involves SQL, dashboards, A/B tests, feature engineering, and stakeholder communication as much as it involves model training. ML is one tool in their toolbox — often the most powerful, but rarely the first one out of the box for a new problem.

## 3. The Three Types of Machine Learning

> **[Picture 1.2 — Machine learning types.]** A taxonomy diagram showing three branches: **Supervised Learning** (with sub-branches *Classification* and *Regression*), **Unsupervised Learning** (with sub-branches *Dimensionality Reduction* and *Clustering*), and **Reinforcement Learning**.

The single most useful question to ask about any ML problem is: *what data do I have?*

- If you have **inputs paired with the correct outputs**, you are doing **supervised learning**.
- If you have **inputs only**, you are doing **unsupervised learning**.
- If you have **no fixed dataset** but you have an **environment that responds to your actions**, you are doing **reinforcement learning**.

Everything else — what algorithm, what loss function, what evaluation metric — flows from this question. Let's now do each of the three in code, on a tiny finance-flavoured problem, to make the distinction concrete.

### 3.1 Supervised Learning

In supervised learning the training set is a collection of input–output pairs:

$$\mathcal{D} \;=\; \{(\mathbf{x}_1, y_1),\; (\mathbf{x}_2, y_2),\; \dots,\; (\mathbf{x}_n, y_n)\}$$

where $\mathbf{x}_i \in \mathbb{R}^p$ is a vector of $p$ features and $y_i$ is the *label* (also called the *target* or *response*). The word **supervised** refers to the fact that during training we have access to the correct $y_i$ for every $\mathbf{x}_i$ — it is as if a teacher is supervising the learning, telling the student which answer is right.

The goal is to learn a function $f_\theta$, parametrised by $\theta$, such that $f_\theta(\mathbf{x}) \approx y$ not just on the training data but on **unseen** data drawn from the same distribution. Formally we minimise an expected loss:

$$\theta^* \;=\; \arg\min_\theta \;\mathbb{E}_{(\mathbf{x}, y)}\bigl[\,\mathcal{L}(f_\theta(\mathbf{x}),\, y)\,\bigr]$$

In practice we approximate the expectation with the training sample — and this is the source of the central tension of ML, the **generalisation problem**: a model that fits training data perfectly may fail on new data.

Supervised learning splits into two cases depending on whether $y$ is categorical or continuous: **classification** and **regression**.

> **[Picture 1.3 — Regression versus classification.]** Two side-by-side scatter plots. *Left:* a regression example where the response is a continuous *return*, with predicted values on the x-axis and observed values on the y-axis falling near a 45-degree line. *Right:* a classification example where the outcome is the categorical label *bull* or *bear*, with points coloured by class and a learned boundary separating them.

#### 3.1.1 Classification — predicting Bull vs Bear regimes

We will build a **classifier** that decides whether the market over the next month will be in a **bull** or **bear** regime, given four synthetic macro features:

- `VIX_change` — change in the implied volatility index
- `Yield_curve` — slope of the Treasury yield curve (10Y minus 2Y)
- `PMI` — Purchasing Managers' Index, a leading indicator of manufacturing activity
- `Credit_spread` — high-yield corporate spread over Treasuries

This is a toy version of the kind of regime-classification model that allocators use to tilt portfolios defensively or offensively.

In [2]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Generate a synthetic dataset that mimics a 4-feature regime-classification problem
X, y = make_classification(
    n_samples=1000, n_features=4, n_informative=3,
    n_redundant=0, n_classes=2, weights=[0.5, 0.5],
    random_state=42
)
feature_names = ['VIX_change', 'Yield_curve', 'PMI', 'Credit_spread']
class_names = ['Bear', 'Bull']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Class distribution (train): Bear={sum(y_train==0)}, Bull={sum(y_train==1)}")

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\nTest accuracy: {acc:.4f}")
print(f"\nLearned coefficients (w):")
for f, c in zip(feature_names, clf.coef_[0]):
    print(f"  {f:>15s}: {c:+.4f}")
print(f"Intercept (b): {clf.intercept_[0]:+.4f}")

Training set: (750, 4), Test set: (250, 4)
Class distribution (train): Bear=378, Bull=372

Test accuracy: 0.9080

Learned coefficients (w):
       VIX_change: +0.1437
      Yield_curve: +3.1810
              PMI: +0.3527
    Credit_spread: +0.8984
Intercept (b): +0.2485


The model is trained on **750** examples and evaluated on **250** held-out examples, with both classes nearly balanced ($378$ Bears vs $372$ Bulls in training). The test accuracy is **$90.80\%$** — well above the $\approx 50\%$ you would expect from random guessing on a balanced binary problem.

**What the model is actually computing.** Logistic regression assigns each example a probability of being class 1 (Bull) by passing a linear combination of the features through the **sigmoid** function:

$$P(y = 1 \mid \mathbf{x}) \;=\; \sigma(\mathbf{w}^T \mathbf{x} + b) \;=\; \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

Each coefficient $w_j$ has a direct interpretation: holding the other features fixed, a one-unit increase in feature $j$ multiplies the **odds** of being Bull by $e^{w_j}$.

**Reading the coefficients.** The yield curve is by far the dominant signal, with $w_{\text{Yield curve}} = +3.18$ — every one-unit increase in curve steepness multiplies the odds of a Bull regime by $e^{3.18} \approx 24$. This is consistent with the empirical regularity that a steepening yield curve precedes economic expansion. The credit spread coefficient is $+0.90$ in this synthetic dataset; in real data we would expect it negative (wider spreads $\rightarrow$ riskier environment $\rightarrow$ more bear-like). The mismatch is a reminder that synthetic data does not encode real economic structure — the synthesis process happened to make this feature positively associated with class 1.

**Production lens.** The huge advantage of logistic regression in finance is that every coefficient is auditable. A regulator can ask "why did you deny this loan?" and the model can answer with a sentence ("your debt-to-income ratio contributed $-1.2$ to the log-odds"). We will see in later chapters that more powerful models (XGBoost, neural networks) often gain only a few accuracy points over a tuned logistic regression — at significant cost in explainability.

#### 3.1.2 Regression — predicting a continuous return

In **regression** the label $y$ is continuous, and we want a real-valued prediction. The canonical financial example is a **factor model**: the return of an asset is modelled as a linear combination of common risk factors plus an asset-specific residual.

$$r_i \;=\; \beta_{i,\text{mkt}} \, r_{\text{mkt}} \;+\; \beta_{i,\text{size}} \, r_{\text{size}} \;+\; \beta_{i,\text{value}} \, r_{\text{value}} \;+\; \varepsilon_i$$

This is a stripped-down Fama–French-style three-factor model. The $\beta$ coefficients tell us how much the asset's return moves with each factor. We will simulate this exact relationship, then ask the model to recover the $\beta$s purely from data.

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(7)
n = 500
market_return = np.random.normal(0.0, 0.012, n)
size_factor   = np.random.normal(0.0, 0.008, n)
value_factor  = np.random.normal(0.0, 0.007, n)
noise         = np.random.normal(0.0, 0.005, n)

# True relationship — a 3-factor model
asset_return = (
    0.85 * market_return
    + 0.30 * size_factor
    - 0.15 * value_factor
    + noise
)

X_reg = np.column_stack([market_return, size_factor, value_factor])
y_reg = asset_return

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

reg = LinearRegression()
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("Coefficients learned by the model:")
print(f"  Beta_market = {reg.coef_[0]:+.4f}  (true: +0.85)")
print(f"  Beta_size   = {reg.coef_[1]:+.4f}  (true: +0.30)")
print(f"  Beta_value  = {reg.coef_[2]:+.4f}  (true: -0.15)")
print(f"  Intercept   = {reg.intercept_:+.5f}")
print(f"\nTest MSE: {mse:.6f}")
print(f"Test RMSE: {np.sqrt(mse):.6f}")
print(f"Test R^2: {r2:.4f}")

Coefficients learned by the model:
  Beta_market = +0.8576  (true: +0.85)
  Beta_size   = +0.3291  (true: +0.30)
  Beta_value  = -0.1391  (true: -0.15)
  Intercept   = -0.00044

Test MSE: 0.000029
Test RMSE: 0.005400
Test R^2: 0.7931


The recovered betas are remarkably close to the true ones: **$+0.8576$ vs $+0.85$** for market, **$+0.3291$ vs $+0.30$** for size, **$-0.1391$ vs $-0.15$** for value. The intercept is essentially zero ($-0.00044$), as it should be — we generated the data with no constant term.

**Loss function.** Linear regression minimises the **mean squared error**,

$$\text{MSE}(\boldsymbol{\beta}, b) \;=\; \frac{1}{n}\sum_{i=1}^{n} \bigl(\,y_i - \mathbf{x}_i^T \boldsymbol{\beta} - b\,\bigr)^2$$

and has the famous closed-form solution $\hat{\boldsymbol{\beta}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$.

**Reading the metrics.** The test MSE is $2.9 \times 10^{-5}$, which becomes a more readable **RMSE of $0.0054$** — about 54 basis points of return error per prediction. The **$R^2 = 0.7931$** tells us the model explains roughly **$79.3\%$ of the variance** in asset returns. The remaining $20.7\%$ is the irreducible idiosyncratic noise we added — even a perfect model could not predict it because, by construction, it is unrelated to any input feature.

**Why this matters.** Most predictive problems in finance have a much lower $R^2$ in the wild — daily equity return predictions typically come in at $R^2 \approx 0.01$–$0.05$, and that is enough to make money if you can scale and trade cheaply. The synthetic $R^2 = 0.79$ here is achievable only because we *know* the data-generating process. In real research the gap between train and test $R^2$ becomes the first thing you check, because a high train $R^2$ paired with a low test $R^2$ is the signature of **overfitting** — a topic we will return to in Chapter 4.

### 3.2 Unsupervised Learning

In **unsupervised learning** the dataset has no labels:

$$\mathcal{D} \;=\; \{\mathbf{x}_1,\; \mathbf{x}_2,\; \dots,\; \mathbf{x}_n\}$$

There is no target $y$ for the algorithm to predict and no "right answer" to score against. Instead, the goal is to discover **structure** that exists in the data — groups of similar examples, low-dimensional directions of variation, or anomalies that don't fit any pattern.

This sounds woolly, but it is the right framing for many problems in finance. You may have transaction logs without fraud labels. You may have a panel of asset returns without knowing how many "regimes" or "factors" drive them. You may want to summarise a $1{,}000$-stock universe into a handful of representative groups. The two unsupervised techniques we cover below — **clustering** and **dimensionality reduction** — are the workhorses for these tasks.

#### 3.2.1 Clustering — grouping stocks by behaviour

The clustering question is: *given a set of points, can we partition them into $k$ groups such that points in the same group are similar and points in different groups are dissimilar?*

We will simulate 90 stocks characterised by two numbers each — annualised return and annualised volatility — drawn from three underlying populations: **defensive** (low return, low vol), **cyclical** (mid return, mid vol), and **growth** (high return, high vol). We hide the population labels from the algorithm and ask **k-means** to discover the grouping on its own.

> **[Picture 1.5 — Clustering.]** A scatter plot showing the entire dataset partitioned into distinct clusters by a clustering algorithm, with each cluster shaded differently. The visual reinforces the idea that clustering finds *natural groupings* in unlabelled data — no class names are given to the algorithm; it infers the groups from geometry alone.

**The k-means objective.** K-means finds $k$ cluster centres $\boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_k$ that minimise the total within-cluster squared distance:

$$\min_{\{\boldsymbol{\mu}_j\}, \{C_j\}} \;\sum_{j=1}^{k} \sum_{\mathbf{x} \in C_j} \|\,\mathbf{x} - \boldsymbol{\mu}_j\,\|_2^2$$

This quantity is reported by scikit-learn as the **inertia** — lower is tighter.

In [4]:
from sklearn.cluster import KMeans

np.random.seed(0)
# Three latent groups of stocks
defensive = np.random.normal([0.04, 0.10], [0.01, 0.02], size=(30, 2))
cyclical  = np.random.normal([0.08, 0.22], [0.02, 0.03], size=(30, 2))
growth    = np.random.normal([0.15, 0.35], [0.03, 0.05], size=(30, 2))
X_cluster = np.vstack([defensive, cyclical, growth])
print(f"Dataset shape: {X_cluster.shape}  "
      "(90 stocks, 2 features: [annual_return, annual_volatility])")

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_cluster)

print(f"\nCluster centers (return, volatility):")
for i, c in enumerate(kmeans.cluster_centers_):
    print(f"  Cluster {i}: return={c[0]:.3f}, volatility={c[1]:.3f}")

print(f"\nInertia (within-cluster sum of squares): {kmeans.inertia_:.5f}")
unique, counts = np.unique(labels, return_counts=True)
print(f"Stocks per cluster: {dict(zip(unique.tolist(), counts.tolist()))}")

Dataset shape: (90, 2)  (90 stocks, 2 features: [annual_return, annual_volatility])

Cluster centers (return, volatility):
  Cluster 0: return=0.082, volatility=0.231
  Cluster 1: return=0.147, volatility=0.361
  Cluster 2: return=0.041, volatility=0.101

Inertia (within-cluster sum of squares): 0.13509
Stocks per cluster: {0: 31, 1: 29, 2: 30}


Without ever seeing a label, k-means has recovered the three underlying groups almost perfectly:

| Discovered cluster | Centre (return, vol)     | True group  | Stocks |
|-------------------:|:-------------------------|:------------|-------:|
| Cluster 2          | $(0.041,\, 0.101)$       | Defensive   | 30     |
| Cluster 0          | $(0.082,\, 0.231)$       | Cyclical    | 31     |
| Cluster 1          | $(0.147,\, 0.361)$       | Growth      | 29     |

The centres match the population means we used to generate the data — $(0.04, 0.10)$, $(0.08, 0.22)$, $(0.15, 0.35)$ — to two decimal places. The stocks per cluster $(31, 29, 30)$ are almost exactly the true $(30, 30, 30)$; only one stock crossed the boundary between defensive and cyclical because of its sampled noise.

The final **inertia** is $0.13509$. Inertia by itself is meaningless — it depends on units and on $k$ — but it is the loss we monitor when choosing the number of clusters via the **elbow method** (plot inertia versus $k$ and look for the kink).

**Why this matters in practice.** A portfolio manager looking at $5{,}000$ stocks does not want to think about $5{,}000$ instruments individually. Clustering compresses the universe into a handful of behavioural groups, and then the manager can think at the group level — *I want 40% in defensive, 35% in cyclical, 25% in growth* — and let portfolio construction handle the within-group choice. The same technique drives customer segmentation in retail banking and risk-bucket assignment in credit.

#### 3.2.2 Dimensionality Reduction — PCA on asset returns

**Dimensionality reduction** is the process of replacing each input vector $\mathbf{x} \in \mathbb{R}^p$ with a lower-dimensional vector $\mathbf{z} \in \mathbb{R}^k$ where $k \ll p$, while preserving as much of the original information as possible.

> **[Picture 1.4 — Dimensionality reduction.]** A scatter plot in $(X_1, X_2)$ space showing data points lying close to a single oblique line, with an arrow projecting them onto that line to give a single coordinate $Z_1$. The figure visualises the central idea: two correlated features carry less than two features' worth of independent information; one well-chosen direction $Z_1$ can capture nearly all of it.

**Principal Component Analysis (PCA)** is the most widely used technique for this. PCA finds the orthogonal directions of maximum variance in the data: the first principal component is the single direction along which the data varies the most; the second is the direction of greatest remaining variance, orthogonal to the first; and so on.

Mathematically, if $\mathbf{X} \in \mathbb{R}^{n \times p}$ is the centred data matrix, the principal components are the eigenvectors of the covariance matrix $\boldsymbol{\Sigma} = \frac{1}{n-1}\mathbf{X}^T\mathbf{X}$, ordered by the size of their eigenvalues:

$$\boldsymbol{\Sigma}\,\mathbf{v}_i \;=\; \lambda_i \mathbf{v}_i, \qquad \lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_p \geq 0$$

The eigenvalue $\lambda_i$ is the variance captured by the $i$-th component, and the *explained variance ratio* is $\lambda_i / \sum_j \lambda_j$.

In finance, PCA is the foundational tool of **factor analysis**: it tells us how many independent sources of risk drive a set of asset returns. We simulate 10 assets driven by 2 latent factors plus idiosyncratic noise, then watch PCA recover them.

In [5]:
from sklearn.decomposition import PCA

np.random.seed(1)
T = 1000
factor_market = np.random.normal(0, 0.012, T)
factor_sector = np.random.normal(0, 0.006, T)
loadings_market = np.random.uniform(0.4, 1.2, 10)
loadings_sector = np.random.uniform(-0.5, 0.8, 10)
idiosync = np.random.normal(0, 0.004, size=(T, 10))

# 10 asset returns driven by 2 latent factors + noise
returns = (
    np.outer(factor_market, loadings_market)
    + np.outer(factor_sector, loadings_sector)
    + idiosync
)
print(f"Returns matrix shape: {returns.shape}  ({T} days, 10 assets)")

pca = PCA(n_components=10)
pca.fit(returns)

print(f"\nExplained variance ratio per component:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v:.4f}  ({v*100:.2f}%)")

cum = np.cumsum(pca.explained_variance_ratio_)
print(f"\nCumulative explained variance:")
for i, v in enumerate(cum):
    print(f"  First {i+1} PC(s): {v:.4f}  ({v*100:.2f}%)")

n_95 = np.searchsorted(cum, 0.95) + 1
print(f"\nComponents needed to explain >=95% variance: {n_95}")

Returns matrix shape: (1000, 10)  (1000 days, 10 assets)

Explained variance ratio per component:
  PC1: 0.8061  (80.61%)
  PC2: 0.0871  (8.71%)
  PC3: 0.0156  (1.56%)
  PC4: 0.0149  (1.49%)
  PC5: 0.0142  (1.42%)
  PC6: 0.0136  (1.36%)
  PC7: 0.0129  (1.29%)
  PC8: 0.0125  (1.25%)
  PC9: 0.0118  (1.18%)
  PC10: 0.0112  (1.12%)

Cumulative explained variance:
  First 1 PC(s): 0.8061  (80.61%)
  First 2 PC(s): 0.8932  (89.32%)
  First 3 PC(s): 0.9088  (90.88%)
  First 4 PC(s): 0.9238  (92.38%)
  First 5 PC(s): 0.9380  (93.80%)
  First 6 PC(s): 0.9516  (95.16%)
  First 7 PC(s): 0.9645  (96.45%)
  First 8 PC(s): 0.9769  (97.69%)
  First 9 PC(s): 0.9888  (98.88%)
  First 10 PC(s): 1.0000  (100.00%)

Components needed to explain >=95% variance: 6


The first two principal components together explain **$89.32\%$** of the total return variance — **$80.61\%$ from PC1 alone** and **$8.71\%$ from PC2**. The remaining eight components together explain just $10.68\%$, distributed roughly evenly across them ($1.1\%$–$1.6\%$ each).

**Recovering the factor structure.** This pattern is exactly what we should expect: we generated the data with two latent factors (a strong market factor with volatility $\sigma_{\text{mkt}} = 0.012$ and a weaker sector factor with $\sigma_{\text{sec}} = 0.006$) plus a small idiosyncratic noise term ($\sigma_{\text{idio}} = 0.004$). The variance ratio of the two factors is approximately

$$\frac{\sigma_{\text{mkt}}^2}{\sigma_{\text{mkt}}^2 + \sigma_{\text{sec}}^2} \;=\; \frac{0.012^2}{0.012^2 + 0.006^2} \;=\; \frac{0.000144}{0.000180} \;\approx\; 80\%$$

which lines up tightly with the **$80.61\%$** PCA assigns to PC1. PCA, given only the 10 noisy return series, has discovered the underlying two-factor structure.

**The compression payoff.** The 10-dimensional return vector for each day can be replaced by a 2-dimensional summary $(z_1, z_2)$ at a cost of only $10.68\%$ of variance. In a real $500$-asset universe, you typically find that **the first 5–10 components explain 70–90% of all return variance** — meaning you can work with a tiny handful of factors instead of hundreds of assets. This is the foundation of **statistical arbitrage**, **risk parity**, and almost every modern portfolio construction technique.

**The $95\%$ threshold.** It takes **6 components** to cross the $95\%$ explained-variance bar. A practitioner choosing $k$ would weigh the bias–variance trade-off: keeping more components captures more variance but reintroduces noise and increases downstream model complexity. We will revisit this trade-off in detail in Chapter 7 on unsupervised learning.

### 3.3 Reinforcement Learning

The third family is fundamentally different from the first two. There is no fixed dataset. Instead there is an **agent** that interacts with an **environment** over time. At each step, the agent observes a state $s_t$, picks an action $a_t$, and receives a reward $r_t$ and a next state $s_{t+1}$. The agent's goal is to learn a **policy** $\pi(a \mid s)$ — a rule for choosing actions — that maximises the expected cumulative reward:

$$\max_{\pi} \;\mathbb{E}_\pi\Bigl[\,\sum_{t=0}^{\infty} \gamma^{\,t} r_t\,\Bigr]$$

where $\gamma \in [0, 1)$ is a **discount factor** that prefers near-term rewards over distant ones.

> **[Picture 1.6 — Reinforcement learning.]** A loop diagram with an *Agent* on one side and an *Environment* on the other. An arrow from agent to environment is labelled **Action**; arrows back from environment to agent are labelled **Reward** and **Observation/State**. The diagram makes the central RL pattern explicit: action $\rightarrow$ feedback $\rightarrow$ updated behaviour, repeating indefinitely.

**How RL differs from supervised learning.** In supervised learning the training data already contains the correct answer for every input. In RL there is no answer key — the agent has to *discover* which actions are good by trying them and observing the consequences. This produces a fundamental tension between **exploration** (trying new actions to learn) and **exploitation** (taking the best action so far based on what you already know).

**Why RL fits finance.** Many financial problems are naturally **sequential** rather than one-shot: trade execution, portfolio rebalancing, market-making, hedging, optimal order placement. In each, your action today changes the state of the world (your inventory, the market impact you have left a footprint of, the time remaining) and constrains the choices available tomorrow. RL is the right tool when the optimal action depends on a sequence of future decisions, not just on a static input vector.

#### A toy RL example — when to sell

We illustrate this with the smallest possible trading problem. You hold **5 lots** of a stock and have **5 minutes** before the market closes. At each minute you choose between **wait** and **sell one lot**. The price per lot drifts down as the day progresses (informed sellers tend to push price down): $\$100, \$99, \$97, \$94, \$90$ at minutes $0$ through $4$. Any lot left at the close is dumped at a steep penalty.

The agent has no idea what the prices will be when it starts. It learns by trial and error, using a tabular **Q-learning** update:

$$Q(s, a) \;\leftarrow\; Q(s, a) \;+\; \alpha\,\Bigl[\,r + \gamma \max_{a'} Q(s', a') - Q(s, a)\,\Bigr]$$

Here $Q(s, a)$ is the agent's current estimate of the long-run reward of taking action $a$ in state $s$; $\alpha$ is the learning rate; and the bracket is the **temporal-difference error** — the gap between the agent's old estimate and a new estimate constructed from the observed reward plus the best future value. Over many episodes, this update converges to the optimal action-value function.

In [6]:
# A minimal Q-learning agent on a 5-minute / 5-lot order-execution task

np.random.seed(123)
n_states  = 6  # lots remaining: 0..5
n_actions = 2  # 0 = wait, 1 = sell one lot
Q = np.zeros((n_states, n_actions))

alpha = 0.1     # learning rate
gamma = 0.95    # discount factor
epsilon = 0.15  # exploration probability
prices = [100, 99, 97, 94, 90]  # price you get per lot if you sell at minute t
n_episodes = 2000

for ep in range(n_episodes):
    lots = 5
    for t in range(5):
        state = lots
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            a = np.random.randint(n_actions)
        else:
            a = int(np.argmax(Q[state]))
        # Step the environment
        if a == 1 and lots > 0:
            reward = prices[t]
            new_lots = lots - 1
        else:
            reward = 0
            new_lots = lots
        new_state = new_lots
        # Q-learning update
        Q[state, a] += alpha * (
            reward + gamma * np.max(Q[new_state]) - Q[state, a]
        )
        lots = new_lots
        if lots == 0:
            break
    # End-of-day penalty for any unsold lots
    if lots > 0:
        Q[lots, :] += alpha * (-50 * lots - Q[lots, :])

print("Learned Q-table (rows = lots_remaining, cols = [wait, sell]):")
print(np.round(Q, 2))

print("\nGreedy policy derived from Q:")
actions_label = ['wait', 'sell']
for s in range(n_states):
    print(f"  lots_remaining={s} -> action: {actions_label[int(np.argmax(Q[s]))]}")

Learned Q-table (rows = lots_remaining, cols = [wait, sell]):
[[  0.     0.  ]
 [-34.23  74.1 ]
 [ 21.33 146.32]
 [ 78.84 219.95]
 [118.89 288.57]
 [229.21 347.97]]

Greedy policy derived from Q:
  lots_remaining=0 -> action: wait
  lots_remaining=1 -> action: sell
  lots_remaining=2 -> action: sell
  lots_remaining=3 -> action: sell
  lots_remaining=4 -> action: sell
  lots_remaining=5 -> action: sell


After **2,000 episodes** of trial and error, the agent has learned a sensible policy: **whenever it still holds inventory, it sells immediately**. Reading the Q-table row by row:

- With **5 lots remaining**, $Q(5, \text{sell}) = 347.97$ vs $Q(5, \text{wait}) = 229.21$ — selling is worth roughly $118$ more units of expected discounted reward than waiting.
- With **1 lot remaining**, $Q(1, \text{sell}) = 74.10$ vs $Q(1, \text{wait}) = -34.23$ — the negative value of waiting reflects the propagated end-of-day penalty: if you wait, you risk getting hit with the $-50$ per unsold lot.
- With **0 lots remaining**, both Q-values are $0$ — there is nothing left to decide.

**Why this policy?** The price schedule $\$100 \to \$99 \to \$97 \to \$94 \to \$90$ falls monotonically, so every minute of delay strictly costs money. Plus there is a heavy terminal penalty ($-50$ per lot) for unsold inventory at close. Given those incentives, the optimal action at every step with inventory remaining is to sell. The agent learned this without ever being told the prices — it discovered them by *acting and observing rewards*. That is the defining feature of RL.

**The path to realism.** A real execution problem has continuous prices, hundreds of time steps, partial fills, market impact (your own selling pushes the price down further), and a state space that includes the order book and recent trades. The tabular Q-learning we used here would not scale to that, which is why production RL systems use **deep Q-networks (DQN)** or **policy-gradient methods** that approximate $Q$ with a neural network. The principle — choose actions to maximize discounted future reward, update on temporal-difference error — is identical. We revisit RL with deep approximators in Chapter 9.

## 4. Natural Language Processing in Finance

**Natural Language Processing (NLP)** is the branch of AI concerned with making machines understand and generate human language. It is not a fourth ML family — it is a *domain* that uses supervised, unsupervised, and increasingly deep learning techniques applied to text.

NLP matters in finance because so much of the relevant information arrives in textual form:

- **Sell-side research reports** — thousands of pages of analyst commentary published every day.
- **Earnings call transcripts** — quarterly, for every public company, with management's words being market-moving.
- **News and social media** — headlines, tweets, Reddit threads, Bloomberg terminal messages.
- **Regulatory filings** — 10-Ks, 10-Qs, prospectuses, with structure but also free-form risk factor disclosures.
- **Internal documents** — emails, chat logs, customer support tickets.

A practitioner's pipeline typically looks like: **clean text $\rightarrow$ tokenize $\rightarrow$ embed $\rightarrow$ aggregate $\rightarrow$ feed to a downstream model**. The "embed" step is where the field has changed most over the last decade: from bag-of-words counts, to TF-IDF, to word2vec/GloVe embeddings, to contextual transformer embeddings such as BERT and its financial variant FinBERT.

Three practical use cases dominate today:

1. **Sentiment scoring** of news and social media as a trading or risk signal.
2. **Information extraction** from filings — pulling structured facts out of free text.
3. **Chatbots and routing** for customer interactions.

**Chapter 10** of this book is dedicated to NLP and contains end-to-end case studies on each of these. For now the only thing to remember is that text is data, and every ML technique you have seen in this chapter has a textual analogue.

## 5. Chapter Summary

We have covered three things in this chapter.

**Where ML is being applied in finance.** Ten distinct application areas — algorithmic trading, portfolio management, fraud detection, underwriting, automation, risk management, asset price prediction, derivative pricing, sentiment analysis, and trade settlement (with AML thrown in). Each uses a different mix of ML techniques, but they share a common pattern: large datasets, repetitive decisions, and a business case where small accuracy improvements translate into large dollar value.

**The terminology.** AI is the goal, ML is the dominant approach, deep learning is a powerful subset of ML, and data science is the practice that pulls all of these together. Being precise about which one you are doing is the first step to communicating clearly with stakeholders.

**The three families of ML.** Supervised learning learns input-output mappings (classification and regression). Unsupervised learning discovers structure in unlabelled data (clustering and dimensionality reduction). Reinforcement learning learns sequential decision policies from interaction with an environment. We built a small finance example for each family — bull/bear classification, factor-model regression, stock-behaviour clustering, PCA on returns, and Q-learning for order execution — to anchor the concepts.

### Key results from the demos

| Demo                           | Family          | Metric                                                    |
|:-------------------------------|:----------------|:----------------------------------------------------------|
| Bull/Bear classification       | Supervised      | Test accuracy **$0.9080$**; yield-curve coefficient $+3.18$ dominates |
| 3-factor regression            | Supervised      | Test $R^2 = 0.7931$, RMSE $= 0.0054$; betas recovered to 2 decimals  |
| K-means on stocks              | Unsupervised    | $3$ clusters recovered exactly $(31, 29, 30)$ vs true $(30, 30, 30)$ |
| PCA on 10 asset returns        | Unsupervised    | PC1 $= 80.61\%$, PC1+PC2 $= 89.32\%$ of total variance              |
| Tabular Q-learning execution   | Reinforcement   | After $2{,}000$ episodes, optimal policy = sell while inventory > 0 |

### What comes next

The next chapter walks through the **end-to-end model development workflow** — data ingestion, exploratory analysis, feature engineering, model selection, validation, and deployment — in a Python-based framework. Every case study in the remainder of the book will follow that workflow, so spend time on it. The single biggest mistake newcomers to ML in finance make is jumping to model selection without first understanding their data; Chapter 2 is the structured cure for that mistake.

## 6. Exercises

Try these before moving on. They consolidate the chapter's concepts and warm up the muscle you will need in Chapter 2.

**Exercise 1 — Place the problem.** For each scenario below, identify (a) the ML family, (b) the sub-task (classification / regression / clustering / dim-reduction / RL), and (c) one challenge specific to financial data:

1. Predicting whether a credit-card transaction is fraudulent, with a history of labelled transactions.
2. Reducing a 500-stock universe to 5 representative groups for portfolio construction.
3. Forecasting tomorrow's closing price of WTI crude oil.
4. Deciding how much of a large order to execute in each 5-minute slice across a trading day to minimize market impact.
5. Building a real-time dashboard summarizing a bank's 200 risk metrics into a handful of summary numbers.

**Exercise 2 — Adapt the regression demo.** Go back to the 3-factor regression in section 3.1.2. Modify the noise standard deviation from $0.005$ to $0.020$ and re-run. What happens to $R^2$? Compute approximately what the $R^2$ should be using $\text{Var}(\text{signal}) / \text{Var}(\text{signal} + \text{noise})$ and compare.

**Exercise 3 — Choose $k$.** In the PCA demo, suppose your downstream model accepts at most $k$ inputs and you want to explain at least $90\%$ of variance. From the cumulative-variance output, what is the smallest $k$ that satisfies this? What if you wanted $99\%$?

**Exercise 4 — Break the RL agent.** In the Q-learning demo, change the price schedule to be *increasing* — say $[90, 94, 97, 99, 100]$ — so that waiting is rewarded. Re-run and inspect the learned policy. Does the agent learn to wait? What if you keep prices flat at $[100, 100, 100, 100, 100]$?

**Exercise 5 — Map the ten applications.** Go through section 1 and, for each of the ten applications, write down which ML family (supervised, unsupervised, reinforcement) and which sub-task you would start with. There is no single right answer — defend your choice in one sentence.